# ResNet18 model

For the ResNet18 model, the configuration and parameters initially utilised were identical to those employed in our homemade CNN. 
The ResNet18 models we made had the following accuracies and the following additions:
- ResNet18 model with the settings of the best cnn model: 19.3%
- ResNet18 model with learning rate set to 0.0005 (previously 0.0001): 28.6%
- ResNet18 model with epochs set to 80 (previously 50) and dropout added: 32.1%
- ResNet18 model with attributes added (factor 2.0): 30.1%
- ResNet18 model  with extra data augmentations: 33.7%



### Imports

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import models, transforms
from PIL import Image
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import StratifiedKFold
from pathlib import Path

from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

#for macbook
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print("Device:", device)


ModuleNotFoundError: No module named 'tensorboard'

### Configs

In [2]:
DATA_DIR = Path("../train_images/train_images")
csv_path = Path("../train_images.csv") 

batch_size = 32
learning_rate = 0.0005 #old value: 0.0001
max_lr = 0.003
weight_decay = 0.0004 
num_epochs = 80
num_classes = 200
val_split = 0.2
seed = 42

torch.manual_seed(seed)
np.random.seed(seed)
print("Device:", device)

# This model is trained on a Apple MPS
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Running on Apple MPS GPU")
else:
    device = torch.device("cpu")
    print("Running on CPU")

NameError: name 'device' is not defined

### Datapreperation

### Dataset class

In [ ]:
all_attributes = np.load("../attributes.npy", allow_pickle=True) #added
num_attributes = all_attributes.shape[1] # added

class BirdDataset(Dataset):
    def __init__(self, csv_file, root_dir, img_col_idx, label_col_idx, attributes_data, transform=None):
        self.data = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.img_col_idx = img_col_idx
        self.label_col_idx = label_col_idx
        self.attributes_data = torch.tensor(attributes_data, dtype=torch.float32) #added
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        filename = str(self.data.iloc[idx, self.img_col_idx])
        clean_filename = filename.lstrip('/').lstrip('\\')
        img_path = os.path.join(self.root_dir, clean_filename)

        try:
            image = Image.open(img_path).convert('RGB')
        except (FileNotFoundError, OSError):
            print(f"Could not open {img_path}, using black image.")
            image = Image.new('RGB', (224, 224), (0, 0, 0))

        # PyTorch labels: 0..199
        raw_label = int(self.data.iloc[idx, self.label_col_idx])
        label = raw_label - 1

        attrs = self.attributes_data[label] #added

        if self.transform:
            image = self.transform(image)

        return image, label, attrs
    
    
    

### Transformers

In [ ]:
#old train transformer

# train_transform = transforms.Compose([
#     transforms.Resize((256, 256)),        
#     transforms.RandomCrop((224, 224)),    #zoom 
#     transforms.RandomHorizontalFlip(p=0.5), 
#     transforms.RandomRotation(degrees=15),  
#     transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02), 
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#     transforms.RandomErasing(p=0.15)
# ])

#new
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),        
    transforms.RandomCrop((224, 224)),    
    transforms.RandomHorizontalFlip(p=0.5), 
    transforms.RandomRotation(degrees=15),  
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406),(0.229, 0.224, 0.225))
])

### Train validation split

In [ ]:
full_train_dataset = BirdDataset('../train_images.csv', '../train_images', 0, 1, attributes_data=all_attributes,transform=train_transform)
full_val_dataset   = BirdDataset('../train_images.csv', '../train_images', 0, 1, attributes_data=all_attributes,transform=val_transform)
labels = full_train_dataset.data.iloc[:, 1].values - 1

skf = StratifiedKFold(n_splits=int(1/val_split), shuffle=True, random_state=seed)
for train_idx, val_idx in skf.split(np.zeros(len(labels)), labels):
    break

train_dataset = Subset(full_train_dataset, train_idx)
val_dataset   = Subset(full_val_dataset, val_idx)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

test_dataset  = BirdDataset('../test_images_path.csv', '../test_images', 1, 2, attributes_data=all_attributes,transform=val_transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Test set: {len(test_dataset)} images")

Train: 3140, Val: 786
Test set: 4000 images


### Model setup

In [45]:
class BirdNet(nn.Module):
    def __init__(self, num_classes=200, num_attributes=0):
        super().__init__()
        self.backbone = models.resnet18(weights=None)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.dropout = nn.Dropout(p=0.5) #added
        self.classifier = nn.Linear(in_features, num_classes)
        
        if num_attributes > 0:
            self.attribute_head = nn.Linear(in_features, num_attributes)
        else:
            self.attribute_head = None

    def forward(self, x):
        feats = self.backbone(x)
        
        feats = self.dropout(feats) #added
        
        if self.attribute_head:
            return self.classifier(feats), self.attribute_head(feats)
        else:
            return self.classifier(feats)

### Attributes

In [ ]:
def get_attribute_weights(dataloader):
    print("Analysing attributes...")
    all_attrs = []
    

    for _, _, attrs in tqdm(dataloader, desc="Scanning weights"):
        all_attrs.append(attrs)
        
    all_attrs = torch.cat(all_attrs, dim=0).float()
    
    freq = all_attrs.mean(dim=0)
    
    weights = (1 - freq) / (freq + 1e-6) 
    
    print(f"Average weight factor: {weights.mean():.2f}")
    return weights

### Training + TensorBoard Logging

In [ ]:
pos_weights = get_attribute_weights(train_loader).to(device)

model = BirdNet(num_classes=200, num_attributes=num_attributes).to(device)
criterion_cls = nn.CrossEntropyLoss(label_smoothing=0.1)
criterion_attr = nn.BCEWithLogitsLoss(pos_weight=pos_weights)
print("Model reset")

#settings
best_acc = 0
writer = SummaryWriter(log_dir="../runs/ResNet18_1") 

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Training Loop
for epoch in range(1, num_epochs+1):
    model.train()
    total_loss, correct, total = 0, 0, 0

    # Progress bar per epoch
    loop = tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs} Training", leave=False)
    """
    for batch_idx, (imgs, labels) in enumerate(loop):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(imgs)
        if isinstance(outputs, tuple):
            outputs = outputs[0]  

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    """
    for batch_idx, (imgs, labels, attrs) in enumerate(loop): # Let op: nu 3 items
        imgs, labels, attrs = imgs.to(device), labels.to(device), attrs.to(device)
        
        optimizer.zero_grad()
        cls_logits, attr_logits = model(imgs) 

        loss_cls = criterion_cls(cls_logits, labels)
        loss_attr = criterion_attr(attr_logits, attrs)

        if batch_idx == 0:
            print(f"Classification Loss: {loss_cls.item():.4f}")
            print(f"Attribute Loss:      {loss_attr.item():.4f}")
        
      
        loss = loss_cls + (2.0 * loss_attr)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * imgs.size(0)
        correct += (cls_logits.argmax(1) == labels).sum().item()
        total += labels.size(0)

        # Batch-level TensorBoard
        step = (epoch - 1) * len(train_loader) + batch_idx
        writer.add_scalar("Loss/train_batch", loss.item(), step)
        writer.add_scalar("Accuracy/train_batch", (cls_logits.argmax(1) == labels).float().mean().item(), step)

        loop.set_postfix(loss=loss.item(), acc=(cls_logits.argmax(1) == labels).float().mean().item())

    train_acc = correct / total
    train_loss = total_loss / total

    # Validation
    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        #for imgs, labels in val_loader:
        for imgs, labels, _ in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            if isinstance(outputs, tuple):
                outputs = outputs[0]
            loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    val_loss = val_loss / total

    # Epoch-level TensorBoard
    writer.add_scalar("Loss/train_epoch", train_loss, epoch)
    writer.add_scalar("Loss/val_epoch", val_loss, epoch)
    writer.add_scalar("Accuracy/train_epoch", train_acc, epoch)
    writer.add_scalar("Accuracy/val_epoch", val_acc, epoch)
    writer.add_scalar("LR", scheduler.get_last_lr()[0], epoch)

    print(f"Epoch {epoch}/{num_epochs} | "
          f"Train Acc: {train_acc:.4f}, Train Loss: {train_loss:.4f} | "
          f"Val Acc: {val_acc:.4f}, Val Loss: {val_loss:.4f}")

    scheduler.step()

    # Save best model
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "../Models/best_model_ResNet18_cnn.pth")
        print("New best model saved")

writer.close()

Bezig met analyseren van attributen dataset...


Scanning weights: 100%|██████████| 99/99 [00:20<00:00,  4.88it/s]


Done! Gemiddelde weight factor: 209.16
Model reset


Epoch 1/80 Training:   0%|          | 0/99 [00:00<?, ?it/s]

Classification Loss: 5.6634
Attribute Loss:      1.3206


...

Classification Loss: 1.5342
Attribute Loss:      1.1587


Epoch 79/80 | Train Acc: 0.9592, Train Loss: 3.5827 | Val Acc: 0.3728, Val Loss: 3.3196


Epoch 80/80 Training:   0%|          | 0/99 [00:00<?, ?it/s]

Classification Loss: 1.3802
Attribute Loss:      1.0595


Epoch 80/80 | Train Acc: 0.9615, Train Loss: 3.5814 | Val Acc: 0.3766, Val Loss: 3.3241


### Test and submission 


In [ ]:
# Loading model
model = BirdNet(num_classes=num_classes, num_attributes=num_attributes).to(device)
checkpoint_path = "../Models/best_model_ResNet18_cnn.pth"

if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f"Loaded model weights from: {checkpoint_path}")
else:
    print(f"file not found: {checkpoint_path}")

model.eval()

# Predictions
predictions = []
print("Starting predictions...")

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        inputs = batch[0].to(device)     
        outputs = model(inputs)
        
        if isinstance(outputs, tuple):
            outputs = outputs[0]
            
        _, preds = torch.max(outputs, 1)
        predictions.extend((preds.cpu().numpy() + 1))

# save as csv
submission = pd.DataFrame({
    "id": range(1, len(predictions) + 1),
    "label": predictions
})

output_file = "../Submission_csv/submissions_ResNet18_cnn.csv"
submission.to_csv(output_file, index=False)

print(f"Subimmision ready: {output_file}")
print(submission.head())

Loaded model weights from: best_model_ResNet18_cnn.pth
Starting predictions...


Predicting: 100%|██████████| 125/125 [00:33<00:00,  3.70it/s]

Subimmision ready: submissions_ResNet18_cnn.csv
   id  label
0   1     67
1   2    104
2   3     79
3   4     12
4   5     74
